In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# load CSVs (timestamps left as-is per instructions)
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_error = pd.read_csv("error_logs.csv")

# ensure value columns are numeric where applicable
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

# timezone object for UTC+8 as required (not applied to timestamps since instruction says do not convert)
tz_utc8 = pytz.timezone('Asia/Shanghai')

# Aggregation helper function
def aggregate_kpi(df, group_cols, value_col='value', top_n=50):
    """
    Aggregates df by group_cols and computes count, min, max, mean, p50, p95 on value_col.
    Returns full aggregated table and top_n sorted by count desc.
    """
    g = df.groupby(group_cols)[value_col]
    agg = g.agg(
        count='count',
        min_value='min',
        max_value='max',
        mean_value='mean',
        p50=lambda x: x.quantile(0.5),
        p95=lambda x: x.quantile(0.95)
    ).reset_index()
    # keep full aggregated table in memory and also return a top-n view
    agg_sorted = agg.sort_values('count', ascending=False)
    top = agg_sorted.head(top_n).copy()
    return agg, top

# Metric aggregation grouped by cmdb_id and kpi_name
metric_agg_full, metric_top = aggregate_kpi(df_metric, ['cmdb_id', 'kpi_name'], 'value', top_n=50)

# Trace aggregation grouped by cmdb_id and trace_name
trace_agg_full, trace_top = aggregate_kpi(df_trace, ['cmdb_id', 'trace_name'], 'value', top_n=50)

# Log aggregation grouped by cmdb_id and log_name
log_agg_full, log_top = aggregate_kpi(df_log, ['cmdb_id', 'log_name'], 'value', top_n=50)

# Error logs: count of messages per cmdb_id and up to 5 example messages per component
# Ensure message column exists
if 'message' not in df_error.columns:
    df_error['message'] = df_error.iloc[:, df_error.columns.get_loc('message')] if 'message' in df_error.columns else None

# Group and aggregate error logs
err_grp = df_error.groupby('cmdb_id')
error_agg_full = err_grp.agg(
    count=('message', 'count'),
    earliest_timestamp=('timestamp', 'min'),
    latest_timestamp=('timestamp', 'max'),
)
# collect up to 5 example messages per component
examples = err_grp['message'].apply(lambda s: s.dropna().astype(str).head(5).tolist()).rename('examples')
error_agg_full = error_agg_full.join(examples)

# prepare compact top view (up to 20 components sorted by count desc)
error_top = error_agg_full.sort_values('count', ascending=False).head(20).reset_index()

# Keep final displayed outputs compact: show selected columns only
metric_top = metric_top[['cmdb_id', 'kpi_name', 'count', 'min_value', 'max_value', 'mean_value', 'p50', 'p95']]
trace_top = trace_top[['cmdb_id', 'trace_name', 'count', 'min_value', 'max_value', 'mean_value', 'p50', 'p95']]
log_top = log_top[['cmdb_id', 'log_name', 'count', 'min_value', 'max_value', 'mean_value', 'p50', 'p95']]
error_top = error_top[['cmdb_id', 'count', 'earliest_timestamp', 'latest_timestamp', 'examples']]

# Display the compact summaries (these variables are stored for later steps as well)
metric_top, trace_top, log_top, error_top

```
Out[1]:
```
# Summary string based on previously computed aggregates (displayed as a single concise summary)
metric_rows = metric_top.shape[0] if 'metric_top' in globals() else 0
trace_rows = trace_top.shape[0] if 'trace_top' in globals() else 0
log_rows = log_top.shape[0] if 'log_top' in globals() else 0
error_rows = error_top.shape[0] if 'error_top' in globals() else 0

# Extract top examples if available
if metric_rows > 0:
    m0 = metric_top.iloc[0]
    metric_example = (
        f"Top metric group: {m0['cmdb_id']} / {m0['kpi_name']} (count={int(m0['count'])}, "
        f"min={m0['min_value']}, p50={m0['p50']}, mean={m0['mean_value']}, p95={m0['p95']}, max={m0['max_value']})"
    )
else:
    metric_example = "No metric groups available."

if trace_rows > 0:
    t0 = trace_top.iloc[0]
    trace_example = (
        f"Top trace group: {t0['cmdb_id']} / {t0['trace_name']} (count={int(t0['count'])}, "
        f"min={t0['min_value']}, p50={t0['p50']}, mean={t0['mean_value']}, p95={t0['p95']}, max={t0['max_value']})"
    )
else:
    trace_example = "No trace groups available."

if log_rows > 0:
    l0 = log_top.iloc[0]
    log_example = (
        f"Top log group: {l0['cmdb_id']} / {l0['log_name']} (count={int(l0['count'])})"
    )
else:
    log_example = "No log groups found (log.csv produced no grouped entries)."

if error_rows > 0:
    e0 = error_top.iloc[0]
    error_example = (
        f"Top error_logs component: {e0['cmdb_id']} (count={int(e0['count'])}, "
        f"earliest={int(e0['earliest_timestamp'])}, latest={int(e0['latest_timestamp'])}, "
        f"examples={e0['examples']})"
    )
else:
    error_example = "No error_logs entries found (error_logs.csv produced no grouped entries)."

summary = (
    f"Summary of telemetry aggregates:\n"
    f"- metric.csv: {metric_rows} groups returned (top groups mostly have count=31). {metric_example}\n"
    f"- trace.csv: {trace_rows} groups returned (top groups mostly have count=30). {trace_example}\n"
    f"- log.csv: {log_rows} groups returned. {log_example}\n"
    f"- error_logs.csv: {error_rows} components with message counts. {error_example}\n\n"
    f"Notes: metric and trace aggregations succeeded and show non-empty top groups. "
    f"No grouped log or error_log results were returned from the files."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                           kpi_name  count     min_value     max_value    mean_value           p50           p95
6         IG01           JVM-Threads_7778_JVM_ThreadCount_Threads     31  8.500000e+01  9.200000e+01  8.770968e+01  8.700000e+01  9.100000e+01
820    Redis01                        OSLinux-CPU_CPU_CPUidleutil     31  7.332280e+01  7.384328e+01  7.359356e+01  7.358680e+01  7.382480e+01
810    Mysql02                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31  6.055678e+06  6.057486e+06  6.056583e+06  6.056583e+06  6.057396e+06
803    Mysql02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount     31  4.000000e+00  4.000000e+00  4.000000e+00  4.000000e+00  4.000000e+00
871    Redis01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
...        ...                                                ...    ...           ...           ...           ...           ...           ...
103       IG02                        OSLinux-CPU_CPU_CPUUserTime     31  2.511427e+01  2.530651e+01  2.517917e+01  2.517140e+01  2.524415e+01
1497  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...     31  4.000000e+00  4.000000e+00  4.000000e+00  4.000000e+00  4.000000e+00
1498  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...     31  5.781400e+04  5.781400e+04  5.781400e+04  5.781400e+04  5.781400e+04
1499  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_Processin...     31  6.831940e+07  6.866643e+07  6.849451e+07  6.848825e+07  6.864769e+07
1500  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...     31  1.095740e+05  1.127000e+05  1.110974e+05  1.110380e+05  1.125240e+05

[50 rows x 8 columns],       cmdb_id                       trace_name  count   min_value   max_value  mean_value         p50         p95
0        IG01         trace.self.duration_mean     30    0.066020    0.733815    0.160816    0.110577    0.554227
1        IG01          trace.self.duration_p95     30    0.245850    2.969600    0.912960    0.803725    2.575337
2        IG01             trace.self.row_count     30  318.000000  576.000000  434.466667  437.000000  507.300000
3        IG01  trace.to_Tomcat01.duration_mean     30    0.045344    0.220137    0.114833    0.097642    0.208149
4        IG01   trace.to_Tomcat01.duration_p95     30    0.110600    1.152350    0.604393    0.607325    1.102680
..        ...                              ...    ...         ...         ...         ...         ...         ...
117  Tomcat01    trace.from_IG02.duration_mean     30    0.058447    0.303527    0.123923    0.113037    0.207938
114  Tomcat01    trace.from_IG01.duration_mean     30    0.045344    0.220137    0.114833    0.097642    0.208149
100      MG02          trace.self.duration_p95     30    0.090550    1.011700    0.357800    0.335075    0.901195
101      MG02             trace.self.row_count     30  271.000000  423.000000  328.800000  331.500000  393.200000
99       MG02         trace.self.duration_mean     30    0.038124    0.140468    0.070272    0.068775    0.110711

[50 rows x 8 columns], Empty DataFrame
Columns: [cmdb_id, log_name, count, min_value, max_value, mean_value, p50, p95]
Index: [], Empty DataFrame
Columns: [cmdb_id, count, earliest_timestamp, latest_timestamp, examples]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Reuse existing loaded DataFrames if present; otherwise load (defensive)
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")
try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")
try:
    df_error
except NameError:
    df_error = pd.read_csv("error_logs.csv")

# Ensure numeric types where applicable (do not convert timestamps)
df_metric['value'] = pd.to_numeric(df_metric.get('value'), errors='coerce')
df_trace['value'] = pd.to_numeric(df_trace.get('value'), errors='coerce')
df_log['value'] = pd.to_numeric(df_log.get('value'), errors='coerce')

# Ensure timestamp numeric for filtering
df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce').astype('Int64')
df_trace['timestamp'] = pd.to_numeric(df_trace['timestamp'], errors='coerce').astype('Int64')
df_log['timestamp'] = pd.to_numeric(df_log['timestamp'], errors='coerce').astype('Int64')
df_error['timestamp'] = pd.to_numeric(df_error['timestamp'], errors='coerce').astype('Int64')

# 1) Calculate global P95 per group (global thresholds) BEFORE filtering
metric_thresholds = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='global_p95')
)

trace_thresholds = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='global_p95')
)

log_thresholds = (
    df_log
    .groupby(['cmdb_id', 'log_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='global_p95')
)

# 2) Filter raw rows to incident window inclusive between '2021-03-04 19:30:00' and '2021-03-04 20:00:00' in UTC+8
tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(datetime.strptime('2021-03-04 19:30:00', '%Y-%m-%d %H:%M:%S'))
end_local = tz.localize(datetime.strptime('2021-03-04 20:00:00', '%Y-%m-%d %H:%M:%S'))
# Convert to epoch seconds (UTC) for comparison with timestamp column which is in seconds
start_epoch = int(start_local.astimezone(pytz.utc).timestamp())
end_epoch = int(end_local.astimezone(pytz.utc).timestamp())

metric_window = df_metric[(df_metric['timestamp'] >= start_epoch) & (df_metric['timestamp'] <= end_epoch)].copy()
trace_window = df_trace[(df_trace['timestamp'] >= start_epoch) & (df_trace['timestamp'] <= end_epoch)].copy()
log_window = df_log[(df_log['timestamp'] >= start_epoch) & (df_log['timestamp'] <= end_epoch)].copy()
error_logs_window = df_error[(df_error['timestamp'] >= start_epoch) & (df_error['timestamp'] <= end_epoch)].copy()

# 3) Metric window exceed analysis
# Join global p95 to window rows
mw = metric_window.merge(metric_thresholds, on=['cmdb_id', 'kpi_name'], how='left')
# Mark exceed (value >= global_p95). If global_p95 is NaN, treat as no exceed.
mw['exceed'] = np.where((~mw['global_p95'].isna()) & (mw['value'] >= mw['global_p95']), True, False)

# Aggregate required stats per group
eps = 1e-9
metric_grouped = (
    mw
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .agg(
        global_p95=('global_p95', 'first'),
        count_in_window=('value', 'count'),
        count_exceed=('exceed', 'sum'),
        earliest_exceed_timestamp=('timestamp', lambda s: int(s[mw.loc[s.index, 'exceed']].min()) if (s.loc[s.index].size>0 and (mw.loc[s.index, 'exceed']).any()) else pd.NA),
        max_value_in_window=('value', 'max')
    )
    .reset_index()
)
# compute max_exceed_ratio
metric_grouped['max_exceed_ratio'] = (
    (metric_grouped['max_value_in_window'] - metric_grouped['global_p95']) /
    (metric_grouped['global_p95'].replace(0, eps))
)
# Keep only groups with count_exceed > 0
metric_exceed_top = (
    metric_grouped[metric_grouped['count_exceed'] > 0]
    .sort_values(['count_exceed', 'max_exceed_ratio'], ascending=[False, False])
    .head(20)
    .loc[:, ['cmdb_id', 'kpi_name', 'global_p95', 'count_in_window', 'count_exceed', 'earliest_exceed_timestamp', 'max_value_in_window', 'max_exceed_ratio']]
)

# 4) Trace window exceed analysis (same logic)
tw = trace_window.merge(trace_thresholds, on=['cmdb_id', 'trace_name'], how='left')
tw['exceed'] = np.where((~tw['global_p95'].isna()) & (tw['value'] >= tw['global_p95']), True, False)

trace_grouped = (
    tw
    .groupby(['cmdb_id', 'trace_name'], dropna=False)
    .agg(
        global_p95=('global_p95', 'first'),
        count_in_window=('value', 'count'),
        count_exceed=('exceed', 'sum'),
        earliest_exceed_timestamp=('timestamp', lambda s: int(s[tw.loc[s.index, 'exceed']].min()) if (s.loc[s.index].size>0 and (tw.loc[s.index, 'exceed']).any()) else pd.NA),
        max_value_in_window=('value', 'max')
    )
    .reset_index()
)
trace_grouped['max_exceed_ratio'] = (
    (trace_grouped['max_value_in_window'] - trace_grouped['global_p95']) /
    (trace_grouped['global_p95'].replace(0, eps))
)

trace_exceed_top = (
    trace_grouped[trace_grouped['count_exceed'] > 0]
    .sort_values(['count_exceed', 'max_exceed_ratio'], ascending=[False, False])
    .head(20)
    .loc[:, ['cmdb_id', 'trace_name', 'global_p95', 'count_in_window', 'count_exceed', 'earliest_exceed_timestamp', 'max_value_in_window', 'max_exceed_ratio']]
)

# 5) Log window handling: detect if numeric values present
log_has_numeric = not log_window['value'].dropna().empty

if log_has_numeric:
    lw = log_window.merge(log_thresholds, on=['cmdb_id', 'log_name'], how='left')
    lw['exceed'] = np.where((~lw['global_p95'].isna()) & (lw['value'] >= lw['global_p95']), True, False)
    log_grouped = (
        lw
        .groupby(['cmdb_id', 'log_name'], dropna=False)
        .agg(
            global_p95=('global_p95', 'first'),
            count_in_window=('value', 'count'),
            count_exceed=('exceed', 'sum'),
            earliest_exceed_timestamp=('timestamp', lambda s: int(s[lw.loc[s.index, 'exceed']].min()) if (s.loc[s.index].size>0 and (lw.loc[s.index, 'exceed']).any()) else pd.NA)
        )
        .reset_index()
    )
    log_exceed_top = (
        log_grouped[log_grouped['count_exceed'] > 0]
        .sort_values(['count_exceed'], ascending=[False])
        .head(20)
        .loc[:, ['cmdb_id', 'log_name', 'global_p95', 'count_in_window', 'count_exceed', 'earliest_exceed_timestamp']]
    )
    log_result = log_exceed_top
else:
    # Non-numeric or empty groups: return counts per cmdb_id within window and up to 5 example log_name entries per cmdb_id
    lg = (
        log_window
        .groupby('cmdb_id', dropna=False)
        .agg(count_in_window=('log_name', 'count'),
             earliest_timestamp=('timestamp', 'min'),
             latest_timestamp=('timestamp', 'max'))
        .reset_index()
    )
    examples = (
        log_window.groupby('cmdb_id', dropna=False)['log_name']
        .apply(lambda s: s.dropna().astype(str).head(5).unique().tolist())
        .reset_index(name='example_log_names')
    )
    log_counts_examples = lg.merge(examples, on='cmdb_id', how='left')
    log_result = log_counts_examples.sort_values('count_in_window', ascending=False).head(20)

# 6) error_logs_window: counts per cmdb_id within window, earliest/latest timestamps, up to 5 example messages
err = (
    error_logs_window
    .groupby('cmdb_id', dropna=False)
    .agg(
        count_in_window=('message', 'count'),
        earliest_timestamp=('timestamp', 'min'),
        latest_timestamp=('timestamp', 'max')
    )
    .reset_index()
)
err_examples = (
    error_logs_window
    .groupby('cmdb_id', dropna=False)['message']
    .apply(lambda s: s.dropna().astype(str).head(5).tolist())
    .reset_index(name='example_messages')
)
error_window_summary = err.merge(err_examples, on='cmdb_id', how='left').sort_values('count_in_window', ascending=False).head(20)

# Keep all intermediate full tables saved in kernel variables (already assigned):
# metric_thresholds, trace_thresholds, log_thresholds, metric_window, trace_window, log_window, error_logs_window

# Final compact outputs to display:
metric_exceed_top, trace_exceed_top, log_result, error_window_summary

```
Out[2]:
```
summary = (
    f"Incident window (2021-03-04 19:30:00 to 20:00:00 UTC+8) summary:\n\n"
    f"1) Metrics (metric.csv): {metric_exceed_top.shape[0]} groups had value >= their global P95 within the window.\n"
    f"   - Many top groups (e.g., IG01, IG02, MG01, MG02, Mysql01) show count_in_window=31 and count_exceed=31,\n"
    f"     but max_exceed_ratio=0.0 for those entries (their max equals the global P95, i.e., no >P95 magnitude).\n"
    f"   - Example top metric: {metric_exceed_top.iloc[0]['cmdb_id']} / {metric_exceed_top.iloc[0]['kpi_name']} "
    f"(global_p95={metric_exceed_top.iloc[0]['global_p95']}, count_in_window={int(metric_exceed_top.iloc[0]['count_in_window'])}, "
    f"count_exceed={int(metric_exceed_top.iloc[0]['count_exceed'])}, earliest_exceed_ts={int(metric_exceed_top.iloc[0]['earliest_exceed_timestamp'])}, "
    f"max_value={metric_exceed_top.iloc[0]['max_value_in_window']}, max_exceed_ratio={metric_exceed_top.iloc[0]['max_exceed_ratio']}).\n\n"
    f"2) Traces (trace.csv): {trace_exceed_top.shape[0]} groups had value >= their global P95 within the window.\n"
    f"   - Examples include dockerA1 trace.self.duration_p95 (global_p95=0.014, count_in_window=30, count_exceed=5, "
    f"earliest_exceed_ts=1614857640, max_value=0.016, max_exceed_ratio≈0.1429) and several row_count-duration groups with 2–3 exceeds.\n\n"
    f"3) Logs (log.csv): no numeric log groups with exceedances were found in the window (result set empty). If log values are non-numeric, "
    f"we instead return counts/examples per component — none were present in the window.\n\n"
    f"4) Error logs (error_logs.csv): no error_log entries were present in the incident window (result set empty).\n\n"
    f"Notes: Global P95 thresholds were computed across the full series before filtering. All reported timestamps are epoch seconds.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name    global_p95  count_in_window  count_exceed  earliest_exceed_timestamp  max_value_in_window  max_exceed_ratio
0       IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09               31            31                 1614857400         2.040136e+09               0.0
3       IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  8.266114e+08               31            31                 1614857400         8.266114e+08               0.0
66      IG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  0.000000e+00               31            31                 1614857400         0.000000e+00               0.0
67      IG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  0.000000e+00               31            31                 1614857400         0.000000e+00               0.0
77      IG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  1.000000e+00               31            31                 1614857400         1.000000e+00               0.0
80      IG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  3.000000e-01               31            31                 1614857400         3.000000e-01               0.0
81      IG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  3.000000e+00               31            31                 1614857400         3.000000e+00               0.0
89      IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09               31            31                 1614857400         2.040136e+09               0.0
92      IG02        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  8.239351e+08               31            31                 1614857400         8.239351e+08               0.0
156     IG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  0.000000e+00               31            31                 1614857400         0.000000e+00               0.0
157     IG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  0.000000e+00               31            31                 1614857400         0.000000e+00               0.0
167     IG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  1.000000e+00               31            31                 1614857400         1.000000e+00               0.0
171     IG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  3.000000e+00               31            31                 1614857400         3.000000e+00               0.0
179     MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09               31            31                 1614857400         2.040136e+09               0.0
182     MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  5.060956e+08               31            31                 1614857400         5.060956e+08               0.0
271     MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09               31            31                 1614857400         2.040136e+09               0.0
572  Mysql01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT  0.000000e+00               31            31                 1614857400         0.000000e+00               0.0
573  Mysql01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT  0.000000e+00               31            31                 1614857400         0.000000e+00               0.0
575  Mysql01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  0.000000e+00               31            31                 1614857400         0.000000e+00               0.0
579  Mysql01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  7.000000e+00               31            31                 1614857400         7.000000e+00               0.0,       cmdb_id                         trace_name  global_p95  count_in_window  count_exceed  earliest_exceed_timestamp  max_value_in_window  max_exceed_ratio
193  dockerA1            trace.self.duration_p95    0.014000               30             5                 1614857640               0.0160          0.142857
211  dockerA2            trace.self.duration_p95    0.012000               30             5                 1614857580               0.0120          0.000000
104      MG02        trace.to_dockerA1.row_count   50.000000               25             3                 1614857460              59.0000          0.180000
188  dockerA1          trace.from_MG02.row_count   50.000000               25             3                 1614857460              59.0000          0.180000
44       MG01      trace.from_dockerA1.row_count   18.000000               25             3                 1614858180              21.0000          0.166667
197  dockerA1            trace.to_MG01.row_count   18.000000               25             3                 1614858180              21.0000          0.166667
77       MG02      trace.from_Tomcat02.row_count   64.000000               30             3                 1614857820              70.0000          0.093750
146  Tomcat02            trace.to_MG02.row_count   64.000000               30             3                 1614857820              70.0000          0.093750
229  dockerB1            trace.self.duration_p95    0.017000               30             3                 1614857460               0.0180          0.058824
14       IG01        trace.to_Tomcat04.row_count   62.000000               30             3                 1614857700              65.0000          0.048387
167  Tomcat04          trace.from_IG01.row_count   62.000000               30             3                 1614857700              65.0000          0.048387
65       MG01        trace.to_dockerA2.row_count   54.000000               25             3                 1614857460              56.0000          0.037037
203  dockerA2          trace.from_MG01.row_count   54.000000               25             3                 1614857460              56.0000          0.037037
17       IG02               trace.self.row_count  520.000000               30             3                 1614857400             530.0000          0.019231
247  dockerB2            trace.self.duration_p95    0.015000               30             3                 1614857520               0.0150          0.000000
43       MG01   trace.from_dockerA1.duration_p95    0.031920               25             2                 1614858000               0.5190         15.259398
196  dockerA1         trace.to_MG01.duration_p95    0.031920               25             2                 1614858000               0.5190         15.259398
87       MG02  trace.from_dockerA2.duration_mean    0.050976               25             2                 1614857820               0.5036          8.879198
216  dockerA2        trace.to_MG02.duration_mean    0.050976               25             2                 1614857820               0.5036          8.879198
88       MG02   trace.from_dockerA2.duration_p95    0.217520               25             2                 1614857820               1.9838          8.120081, Empty DataFrame
Columns: [cmdb_id, count_in_window, earliest_timestamp, latest_timestamp, example_log_names]
Index: [], Empty DataFrame
Columns: [cmdb_id, count_in_window, earliest_timestamp, latest_timestamp, example_messages]
Index: [])```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse existing trace_exceed_top if available; otherwise reconstruct from trace_window and trace_thresholds
try:
    trace_exceed_top
except NameError:
    # Reconstruct (should rarely be needed since previous cell created trace_exceed_top)
    tw = trace_window.merge(trace_thresholds, on=['cmdb_id', 'trace_name'], how='left')
    tw['exceed'] = np.where((~tw['global_p95'].isna()) & (tw['value'] >= tw['global_p95']), True, False)
    eps = 1e-9
    trace_grouped = (
        tw
        .groupby(['cmdb_id', 'trace_name'], dropna=False)
        .agg(
            global_p95=('global_p95', 'first'),
            count_in_window=('value', 'count'),
            count_exceed=('exceed', 'sum'),
            earliest_exceed_timestamp=('timestamp', lambda s: int(s[tw.loc[s.index, 'exceed']].min()) if (s.size>0 and tw.loc[s.index, 'exceed'].any()) else pd.NA),
            max_value_in_window=('value', 'max')
        )
        .reset_index()
    )
    trace_grouped['max_exceed_ratio'] = (
        (trace_grouped['max_value_in_window'] - trace_grouped['global_p95']) /
        (trace_grouped['global_p95'].replace(0, eps))
    )
    trace_exceed_top = trace_grouped[trace_grouped['count_exceed'] > 0]

# Filter for severe exceedances (max_exceed_ratio >= 1.0), sort and limit to top 20
trace_severe_top = (
    trace_exceed_top[trace_exceed_top['max_exceed_ratio'] >= 1.0]
    .sort_values('max_exceed_ratio', ascending=False)
    .head(20)
    .loc[:, ['cmdb_id', 'trace_name', 'global_p95', 'count_in_window', 'count_exceed', 'earliest_exceed_timestamp', 'max_value_in_window', 'max_exceed_ratio']]
)

# Display the compact result table
trace_severe_top

```
Out[3]:
```
# Compose a plain-English summary from the previously computed trace_severe_top table
try:
    df = trace_severe_top
except NameError:
    df = None

if df is None or df.empty:
    summary = "No trace groups with max_exceed_ratio >= 1.0 were found in the incident window."
else:
    lines = []
    lines.append(f"Severe trace exceedances (max_exceed_ratio >= 1.0) within the incident window: {len(df)} group(s).")
    lines.append("") 
    for i, r in enumerate(df.itertuples(index=False), start=1):
        cmdb_id = r.cmdb_id
        name = r.trace_name
        gp95 = r.global_p95
        cnt = int(r.count_in_window) if pd.notna(r.count_in_window) else r.count_in_window
        cnt_exc = int(r.count_exceed) if pd.notna(r.count_exceed) else r.count_exceed
        earliest = int(r.earliest_exceed_timestamp) if pd.notna(r.earliest_exceed_timestamp) else r.earliest_exceed_timestamp
        maxv = r.max_value_in_window
        ratio = r.max_exceed_ratio
        lines.append(
            f"{i}) {cmdb_id} - {name}: global_p95={gp95:.6g}, count_in_window={cnt}, "
            f"count_exceed={cnt_exc}, earliest_exceed_ts={earliest}, max_value={maxv:.6g}, "
            f"max_exceed_ratio={ratio:.6g}"
        )
    lines.append("") 
    lines.append("Notes: All listed groups had exactly 2 exceedances during the window; the highest severity is the MG01/dockerA1 pair (max_exceed_ratio ≈ 15.26).")
    summary = "\n".join(lines)

# Display the plain-English summary string (compact)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                         trace_name  global_p95  count_in_window  count_exceed  earliest_exceed_timestamp  max_value_in_window  max_exceed_ratio
43       MG01   trace.from_dockerA1.duration_p95    0.031920               25             2                 1614858000               0.5190         15.259398
196  dockerA1         trace.to_MG01.duration_p95    0.031920               25             2                 1614858000               0.5190         15.259398
87       MG02  trace.from_dockerA2.duration_mean    0.050976               25             2                 1614857820               0.5036          8.879198
216  dockerA2        trace.to_MG02.duration_mean    0.050976               25             2                 1614857820               0.5036          8.879198
88       MG02   trace.from_dockerA2.duration_p95    0.217520               25             2                 1614857820               1.9838          8.120081```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime, timezone

# Reuse existing kernel variables where possible
try:
    trace_window
    trace_thresholds
    metric_window
    metric_thresholds
    log_window
    error_logs_window
except NameError as e:
    raise RuntimeError("Required kernel tables are missing: " + str(e))

# Setup
tz = pytz.timezone('Asia/Shanghai')
spikes = [1614857820, 1614858000]
allowed_components = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
                      "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]
eps = 1e-9

# Helper to add datetime_utc8 column (string)
def add_datetime_utc8(df, ts_col='timestamp'):
    if ts_col not in df.columns:
        df = df.copy()
        df[ts_col] = pd.Series(dtype='Int64')
    df = df.copy()
    # create tz-aware datetimes from epoch seconds (assume timestamps are UTC epoch seconds)
    dt = pd.to_datetime(df[ts_col].astype('Int64').astype('int64'), unit='s', utc=True)
    dt_local = dt.dt.tz_convert(tz)
    df['datetime_utc8'] = dt_local.dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    return df

# 1) Trace context: rows exactly at the two spike timestamps
trace_context_rows = trace_window[trace_window['timestamp'].isin(spikes)].copy()
if not trace_context_rows.empty:
    trace_context_rows = add_datetime_utc8(trace_context_rows, 'timestamp')
    trace_context_rows = trace_context_rows.loc[:, ['timestamp', 'datetime_utc8', 'cmdb_id', 'trace_name', 'value']]
    trace_context_rows = trace_context_rows.sort_values(['timestamp', 'value'], ascending=[True, False]).head(50)
else:
    trace_context_rows = pd.DataFrame(columns=['timestamp', 'datetime_utc8', 'cmdb_id', 'trace_name', 'value'])

# 2) Metric snapshot for allowed components at each spike timestamp
m_snap_list = []
for s in spikes:
    mv = metric_window[(metric_window['timestamp'] == s) & (metric_window['cmdb_id'].isin(allowed_components))].copy()
    if mv.empty:
        continue
    # join global_p95 from metric_thresholds
    mv = mv.merge(metric_thresholds, left_on=['cmdb_id', 'kpi_name'], right_on=['cmdb_id', 'kpi_name'], how='left')
    # ensure numeric
    mv['value'] = pd.to_numeric(mv['value'], errors='coerce')
    mv['global_p95'] = pd.to_numeric(mv['global_p95'], errors='coerce')
    mv['exceed'] = (~mv['global_p95'].isna()) & (mv['value'] >= mv['global_p95'])
    mv['exceed_ratio'] = (mv['value'] - mv['global_p95']) / mv['global_p95'].replace(0, eps)
    mv = add_datetime_utc8(mv, 'timestamp')
    mv = mv.loc[:, ['timestamp', 'datetime_utc8', 'cmdb_id', 'kpi_name', 'value', 'global_p95', 'exceed', 'exceed_ratio']]
    mv = mv[mv['exceed'] == True]
    if not mv.empty:
        m_snap_list.append(mv)

if m_snap_list:
    metric_exceed_snapshot = pd.concat(m_snap_list, ignore_index=True)
    metric_exceed_snapshot = metric_exceed_snapshot.sort_values(['timestamp', 'exceed_ratio'], ascending=[True, False]).head(50)
else:
    metric_exceed_snapshot = pd.DataFrame(columns=['timestamp', 'datetime_utc8', 'cmdb_id', 'kpi_name', 'value', 'global_p95', 'exceed', 'exceed_ratio'])

# 3) Logs and error logs around spikes (+/- 60 seconds), for allowed components
logs_near_7820 = log_window[
    (log_window['timestamp'] >= (spikes[0] - 60)) & (log_window['timestamp'] <= (spikes[0] + 60)) &
    (log_window['cmdb_id'].isin(allowed_components))
].copy()
if not logs_near_7820.empty:
    logs_near_7820 = add_datetime_utc8(logs_near_7820, 'timestamp')
    # select columns: timestamp, datetime_utc8, cmdb_id, log_name, value
    cols = [c for c in ['timestamp', 'datetime_utc8', 'cmdb_id', 'log_name', 'value'] if c in logs_near_7820.columns]
    logs_near_7820 = logs_near_7820.loc[:, cols].sort_values(['timestamp'], ascending=True).head(20)
else:
    logs_near_7820 = pd.DataFrame(columns=['timestamp', 'datetime_utc8', 'cmdb_id', 'log_name', 'value'])

logs_near_8000 = log_window[
    (log_window['timestamp'] >= (spikes[1] - 60)) & (log_window['timestamp'] <= (spikes[1] + 60)) &
    (log_window['cmdb_id'].isin(allowed_components))
].copy()
if not logs_near_8000.empty:
    logs_near_8000 = add_datetime_utc8(logs_near_8000, 'timestamp')
    cols = [c for c in ['timestamp', 'datetime_utc8', 'cmdb_id', 'log_name', 'value'] if c in logs_near_8000.columns]
    logs_near_8000 = logs_near_8000.loc[:, cols].sort_values(['timestamp'], ascending=True).head(20)
else:
    logs_near_8000 = pd.DataFrame(columns=['timestamp', 'datetime_utc8', 'cmdb_id', 'log_name', 'value'])

errors_near_7820 = error_logs_window[
    (error_logs_window['timestamp'] >= (spikes[0] - 60)) & (error_logs_window['timestamp'] <= (spikes[0] + 60)) &
    (error_logs_window['cmdb_id'].isin(allowed_components))
].copy()
if not errors_near_7820.empty:
    errors_near_7820 = add_datetime_utc8(errors_near_7820, 'timestamp')
    cols = [c for c in ['timestamp', 'datetime_utc8', 'cmdb_id', 'message'] if c in errors_near_7820.columns]
    errors_near_7820 = errors_near_7820.loc[:, cols].sort_values(['timestamp'], ascending=True).head(20)
else:
    errors_near_7820 = pd.DataFrame(columns=['timestamp', 'datetime_utc8', 'cmdb_id', 'message'])

errors_near_8000 = error_logs_window[
    (error_logs_window['timestamp'] >= (spikes[1] - 60)) & (error_logs_window['timestamp'] <= (spikes[1] + 60)) &
    (error_logs_window['cmdb_id'].isin(allowed_components))
].copy()
if not errors_near_8000.empty:
    errors_near_8000 = add_datetime_utc8(errors_near_8000, 'timestamp')
    cols = [c for c in ['timestamp', 'datetime_utc8', 'cmdb_id', 'message'] if c in errors_near_8000.columns]
    errors_near_8000 = errors_near_8000.loc[:, cols].sort_values(['timestamp'], ascending=True).head(20)
else:
    errors_near_8000 = pd.DataFrame(columns=['timestamp', 'datetime_utc8', 'cmdb_id', 'message'])

# Keep full intermediate tables unchanged in kernel variables as requested.
# Final compact outputs (only these will be displayed)
trace_context_rows, metric_exceed_snapshot, logs_near_7820, logs_near_8000, errors_near_7820, errors_near_8000

```
Out[4]:
```
import pandas as pd

# Build a concise plain-English summary using existing kernel tables
try:
    trace_df = trace_context_rows
except NameError:
    trace_df = pd.DataFrame()

try:
    metric_df = metric_exceed_snapshot
except NameError:
    metric_df = pd.DataFrame()

try:
    logs_7820 = logs_near_7820
except NameError:
    logs_7820 = pd.DataFrame()

try:
    logs_8000 = logs_near_8000
except NameError:
    logs_8000 = pd.DataFrame()

try:
    errs_7820 = errors_near_7820
except NameError:
    errs_7820 = pd.DataFrame()

try:
    errs_8000 = errors_near_8000
except NameError:
    errs_8000 = pd.DataFrame()

lines = []
lines.append("Summary for spikes at 1614857820 and 1614858000 (UTC+8):")
lines.append("")

# 1) Trace context
n_trace = len(trace_df)
if n_trace == 0:
    lines.append("1) Trace context: empty — no trace_window rows exactly at the spike timestamps.")
else:
    lines.append(f"1) Trace context: {n_trace} trace rows found exactly at the spike timestamps.")
    # show up to 3 representative rows
    sample = trace_df.sort_values(['timestamp','value'], ascending=[True, False]).head(3)
    sample_lines = []
    for r in sample.itertuples(index=False):
        sample_lines.append(f"- {r.timestamp} / {r.datetime_utc8}: {r.cmdb_id} - {r.trace_name} = {r.value}")
    lines.extend(sample_lines)
lines.append("")

# 2) Metric snapshot for allowed root-cause components
n_metric = len(metric_df)
if n_metric == 0:
    lines.append("2) Metric snapshot (allowed components): no metric rows exceeded their global P95 at the spike timestamps.")
else:
    lines.append(f"2) Metric snapshot (allowed components): {n_metric} metric exceed rows found at the spike timestamps.")
    # show top 3 by exceed_ratio
    sample = metric_df.sort_values(['timestamp','exceed_ratio'], ascending=[True, False]).head(3)
    for r in sample.itertuples(index=False):
        lines.append(f"- {r.timestamp} / {r.datetime_utc8}: {r.cmdb_id} - {r.kpi_name}: value={r.value}, global_p95={r.global_p95}, exceed_ratio={r.exceed_ratio:.6g}")
lines.append("")

# 3) Logs and error logs around spikes (+/-60s)
def summarize_window(df, name, spike_ts):
    if df.empty:
        return f"{name} around {spike_ts}: empty."
    else:
        cnt = len(df)
        # show up to 2 example rows
        ex = df.sort_values('timestamp').head(2)
        ex_lines = []
        for r in ex.itertuples(index=False):
            if 'log_name' in df.columns:
                ex_lines.append(f"{r.timestamp} / {r.datetime_utc8}: {r.cmdb_id} - {getattr(r, 'log_name', '')} = {getattr(r, 'value', '')}")
            else:
                ex_lines.append(f"{r.timestamp} / {r.datetime_utc8}: {r.cmdb_id} - message = {getattr(r, 'message', '')}")
        return f"{name} around {spike_ts}: {cnt} rows. Examples: " + " | ".join(ex_lines)

lines.append("3) Logs and error logs within ±60s of each spike (allowed components):")
lines.append(summarize_window(logs_7820, "logs", 1614857820))
lines.append(summarize_window(errs_7820, "error_logs", 1614857820))
lines.append(summarize_window(logs_8000, "logs", 1614858000))
lines.append(summarize_window(errs_8000, "error_logs", 1614858000))
lines.append("")

lines.append("Notes:")
lines.append("- Trace context shows many high row_count traces at 1614857820 (examples include Tomcat03, dockerB1, dockerB2, Tomcat02).")
lines.append("- Severe trace-related spikes previously identified (e.g., MG01/dockerA1 and MG02/dockerA2) are reflected in trace data; metric exceedances present but often small ratios except a few notable CPU/disk/network metrics.")
lines.append("- No logs or error log entries were found around these spikes for the allowed components (empty results).")

summary = "\n".join(lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(       timestamp            datetime_utc8   cmdb_id                     trace_name  value
1550  1614857820  2021-03-04 19:37:00 CST  Tomcat03           trace.self.row_count  982.0
1610  1614857820  2021-03-04 19:37:00 CST  dockerB1           trace.self.row_count  953.0
1625  1614857820  2021-03-04 19:37:00 CST  dockerB2           trace.self.row_count  935.0
1535  1614857820  2021-03-04 19:37:00 CST  Tomcat02           trace.self.row_count  890.0
1565  1614857820  2021-03-04 19:37:00 CST  Tomcat04           trace.self.row_count  825.0
...          ...                      ...       ...                            ...    ...
1496  1614857820  2021-03-04 19:37:00 CST      MG02  trace.from_dockerB2.row_count   56.0
1526  1614857820  2021-03-04 19:37:00 CST  Tomcat01        trace.to_MG02.row_count   56.0
1631  1614857820  2021-03-04 19:37:00 CST  dockerB2        trace.to_MG02.row_count   56.0
1436  1614857820  2021-03-04 19:37:00 CST      MG01  trace.from_Tomcat01.row_count   53.0
1523  1614857820  2021-03-04 19:37:00 CST  Tomcat01        trace.to_MG01.row_count   53.0

[50 rows x 5 columns],       timestamp            datetime_utc8  cmdb_id                                           kpi_name        value     global_p95  exceed  exceed_ratio
31   1614857820  2021-03-04 19:37:00 CST     IG02                    OSLinux-CPU_CPU-2_SingleCpuidle      91.4547      49.963407    True      0.830434
151  1614857820  2021-03-04 19:37:00 CST  Redis01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite      22.4000      20.533310    True      0.090910
162  1614857820  2021-03-04 19:37:00 CST  Redis01    OSLinux-OSLinux_NETWORK_ens160_NETKBTotalPerSec  768155.0000  721148.600000    True      0.065183
124  1614857820  2021-03-04 19:37:00 CST  Mysql02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKWTps     306.8333     288.113300    True      0.064974
125  1614857820  2021-03-04 19:37:00 CST  Mysql02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKWrite    4071.4667    3849.253345    True      0.057729
..          ...                      ...      ...                                                ...          ...            ...     ...           ...
35   1614857820  2021-03-04 19:37:00 CST     IG02  OSLinux-OSLinux_FILESYSTEM_-boot_FSAvailableSpace     768.0000     768.000000    True      0.000000
36   1614857820  2021-03-04 19:37:00 CST     IG02        OSLinux-OSLinux_FILESYSTEM_-boot_FSCapacity      15.4170      15.417000    True      0.000000
37   1614857820  2021-03-04 19:37:00 CST     IG02  OSLinux-OSLinux_FILESYSTEM_-boot_FSInodeUsedPe...       0.4715       0.471500    True      0.000000
38   1614857820  2021-03-04 19:37:00 CST     IG02       OSLinux-OSLinux_FILESYSTEM_-boot_FSUsedSpace     140.0000     140.000000    True      0.000000
39   1614857820  2021-03-04 19:37:00 CST     IG02  OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSAvail...    3730.0000    3730.000000    True      0.000000

[50 rows x 8 columns], Empty DataFrame
Columns: [timestamp, datetime_utc8, cmdb_id, log_name, value]
Index: [], Empty DataFrame
Columns: [timestamp, datetime_utc8, cmdb_id, log_name, value]
Index: [], Empty DataFrame
Columns: [timestamp, datetime_utc8, cmdb_id, message]
Index: [], Empty DataFrame
Columns: [timestamp, datetime_utc8, cmdb_id, message]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Reuse kernel tables: metric_window, metric_thresholds, trace_window, log_window, error_logs_window
# Define helper to add datetime_utc8 (reuse if already present)
tz = pytz.timezone('Asia/Shanghai')
def add_datetime_utc8(df, ts_col='timestamp'):
    df = df.copy()
    if df.empty:
        df['datetime_utc8'] = pd.Series(dtype=str)
        return df
    # ensure integer epoch seconds
    ts = df[ts_col].astype('Int64').astype('int64')
    dt = pd.to_datetime(ts, unit='s', utc=True)
    df['datetime_utc8'] = dt.dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
    return df

# Parameters
spike1 = 1614858000  # for MG01
spike2 = 1614857820  # for MG02
eps = 1e-9

# 1) Metrics: filter metric_window for the two (timestamp, cmdb_id) pairs, join thresholds, compute exceed & ratio
metric_rows = metric_window[
    ((metric_window['timestamp'] == spike1) & (metric_window['cmdb_id'] == 'MG01')) |
    ((metric_window['timestamp'] == spike2) & (metric_window['cmdb_id'] == 'MG02'))
].copy()

if not metric_rows.empty:
    metric_rows = metric_rows.merge(metric_thresholds, on=['cmdb_id','kpi_name'], how='left')
    metric_rows['value'] = pd.to_numeric(metric_rows['value'], errors='coerce')
    metric_rows['global_p95'] = pd.to_numeric(metric_rows['global_p95'], errors='coerce')
    metric_rows['exceed'] = (~metric_rows['global_p95'].isna()) & (metric_rows['value'] >= metric_rows['global_p95'])
    metric_rows['exceed_ratio'] = (metric_rows['value'] - metric_rows['global_p95']) / metric_rows['global_p95'].replace(0, eps)
    metric_exceed_rows = add_datetime_utc8(metric_rows, 'timestamp')
    metric_exceed_rows = metric_exceed_rows[metric_exceed_rows['exceed']].loc[:, 
        ['timestamp','datetime_utc8','cmdb_id','kpi_name','value','global_p95','exceed_ratio']]
    metric_exceed_rows = metric_exceed_rows.sort_values(['timestamp','exceed_ratio'], ascending=[True, False]).head(50)
else:
    metric_exceed_rows = pd.DataFrame(columns=['timestamp','datetime_utc8','cmdb_id','kpi_name','value','global_p95','exceed_ratio'])

# 2) Traces: for MG01 at spike1 and MG02 at spike2 return trace_window rows
trace_specific_rows = trace_window[
    ((trace_window['timestamp'] == spike1) & (trace_window['cmdb_id'] == 'MG01')) |
    ((trace_window['timestamp'] == spike2) & (trace_window['cmdb_id'] == 'MG02'))
].copy()

if not trace_specific_rows.empty:
    trace_specific_rows = add_datetime_utc8(trace_specific_rows, 'timestamp')
    trace_specific_rows = trace_specific_rows.loc[:, ['timestamp','datetime_utc8','cmdb_id','trace_name','value']]
    trace_specific_rows = trace_specific_rows.sort_values(['timestamp','value'], ascending=[True, False]).head(50)
else:
    trace_specific_rows = pd.DataFrame(columns=['timestamp','datetime_utc8','cmdb_id','trace_name','value'])

# 3) Logs/Error counts for MG01 and MG02 within ±60s of each spike, plus up to 5 example rows if present
components = ['MG01','MG02']
spikes = [spike1, spike2]

rows = []
logs_examples_list = []
errors_examples_list = []

for sp in spikes:
    start = sp - 60
    end = sp + 60
    for comp in components:
        lw = log_window[
            (log_window['cmdb_id'] == comp) &
            (log_window['timestamp'] >= start) & (log_window['timestamp'] <= end)
        ].copy()
        ew = error_logs_window[
            (error_logs_window['cmdb_id'] == comp) &
            (error_logs_window['timestamp'] >= start) & (error_logs_window['timestamp'] <= end)
        ].copy()
        log_count = 0 if lw.empty else int(len(lw))
        err_count = 0 if ew.empty else int(len(ew))
        rows.append({'spike_timestamp': sp, 'cmdb_id': comp, 'log_count': log_count, 'error_log_count': err_count})
        # examples up to 5
        if not lw.empty:
            ex = add_datetime_utc8(lw, 'timestamp').loc[:, ['timestamp','datetime_utc8','cmdb_id','log_name','value']].head(5)
            ex['spike_timestamp'] = sp
            logs_examples_list.append(ex)
        if not ew.empty:
            ex2 = add_datetime_utc8(ew, 'timestamp').loc[:, ['timestamp','datetime_utc8','cmdb_id','message']].head(5)
            ex2['spike_timestamp'] = sp
            errors_examples_list.append(ex2)

log_error_counts = pd.DataFrame(rows).sort_values(['spike_timestamp','cmdb_id']).reset_index(drop=True)

if logs_examples_list:
    logs_examples = pd.concat(logs_examples_list, ignore_index=True).loc[:, ['spike_timestamp','timestamp','datetime_utc8','cmdb_id','log_name','value']]
else:
    logs_examples = pd.DataFrame(columns=['spike_timestamp','timestamp','datetime_utc8','cmdb_id','log_name','value'])

if errors_examples_list:
    errors_examples = pd.concat(errors_examples_list, ignore_index=True).loc[:, ['spike_timestamp','timestamp','datetime_utc8','cmdb_id','message']]
else:
    errors_examples = pd.DataFrame(columns=['spike_timestamp','timestamp','datetime_utc8','cmdb_id','message'])

# Final compact outputs (do not print raw series)
metric_exceed_rows, trace_specific_rows, log_error_counts, logs_examples, errors_examples

```
Out[5]:
```
Summary (focused on MG02 @ 1614857820 and MG01 @ 1614858000):

1) Metrics (exceedances)
- Found 35 metric rows where value >= global P95 for the two queries combined (rows returned for MG02 at 1614857820 and MG01 at 1614858000).
- Many of these metric exceedances have exceed_ratio = 0.0 (value == global_p95), i.e., they hit the P95 threshold but did not substantially exceed it. A few entries show small positive ratios (minor exceedances).

2) Traces (exact timestamps)
- MG02 @ 1614857820: prominent trace spikes include
  - trace.self.row_count = 423
  - trace.from_dockerB1.row_count = 81
  - trace.from_Tomcat03.row_count = 77
  - trace.from_Tomcat02.row_count = 70
  - trace.from_Tomcat04.row_count = 57
- MG01 @ 1614858000: prominent trace spikes include
  - trace.from_Tomcat02.row_count = 42
  - trace.from_Tomcat01.row_count = 39
  - trace.to_dockerB1.row_count = 39
  - trace.from_dockerB1.row_count = 33
  - trace.from_dockerB2.row_count = 27
- Interpretation: both spikes are characterized by elevated row_count/request counts on MG02 (19:37) and MG01 (19:40), suggesting traffic/throughput surges in those message-gateway components.

3) Logs and error logs (±60s windows)
- For both MG01 and MG02 at both spike times, log_window and error_logs_window contain zero rows in the ±60s windows (log_count = 0 and error_log_count = 0). No example log or error messages were found to correlate with these spikes.

Overall conclusion
- The two severe timestamps correspond to trace-level row_count spikes on MG02 (19:37 UTC+8) and MG01 (19:40 UTC+8). Metrics show multiple groups at or just at/above their global P95 thresholds, but most metric exceedances are equality to P95 (no large metric deltas). No logs or error messages were present around the spikes to provide additional clues.

The original code execution output of IPython Kernel is also provided below for reference:

(     timestamp            datetime_utc8 cmdb_id                                           kpi_name         value    global_p95  exceed_ratio
0   1614857820  2021-03-04 19:37:00 CST    MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09           0.0
4   1614857820  2021-03-04 19:37:00 CST    MG02          JVM-Operating System_7779_JVM_JVM_CPULoad  2.579050e+01  2.579050e+01           0.0
9   1614857820  2021-03-04 19:37:00 CST    MG02                    OSLinux-CPU_CPU-3_SingleCpuUtil  1.000000e+02  1.000000e+02           0.0
17  1614857820  2021-03-04 19:37:00 CST    MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKAvg...  8.300000e-01  8.300000e-01           0.0
18  1614857820  2021-03-04 19:37:00 CST    MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKAvg...  9.500000e-01  9.500000e-01           0.0
..         ...                      ...     ...                                                ...           ...           ...           ...
74  1614858000  2021-03-04 19:40:00 CST    MG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMemPerc  0.000000e+00  0.000000e+00           0.0
75  1614858000  2021-03-04 19:40:00 CST    MG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  3.000000e-01  3.000000e-01           0.0
76  1614858000  2021-03-04 19:40:00 CST    MG01    OSLinux-OSLinux_SWAP_SWAP_SWPTotSwapUsedPercent  0.000000e+00  0.000000e+00           0.0
77  1614858000  2021-03-04 19:40:00 CST    MG01                       OSLinux-OSLinux_SWAP_SWAP_Si  0.000000e+00  0.000000e+00           0.0
78  1614858000  2021-03-04 19:40:00 CST    MG01                       OSLinux-OSLinux_SWAP_SWAP_So  0.000000e+00  0.000000e+00           0.0

[35 rows x 7 columns],        timestamp            datetime_utc8 cmdb_id                     trace_name  value
1499  1614857820  2021-03-04 19:37:00 CST    MG02           trace.self.row_count  423.0
1493  1614857820  2021-03-04 19:37:00 CST    MG02  trace.from_dockerB1.row_count   81.0
1481  1614857820  2021-03-04 19:37:00 CST    MG02  trace.from_Tomcat03.row_count   77.0
1478  1614857820  2021-03-04 19:37:00 CST    MG02  trace.from_Tomcat02.row_count   70.0
1484  1614857820  2021-03-04 19:37:00 CST    MG02  trace.from_Tomcat04.row_count   57.0
...          ...                      ...     ...                            ...    ...
2123  1614858000  2021-03-04 19:40:00 CST    MG01  trace.from_Tomcat02.row_count   42.0
2120  1614858000  2021-03-04 19:40:00 CST    MG01  trace.from_Tomcat01.row_count   39.0
2153  1614858000  2021-03-04 19:40:00 CST    MG01    trace.to_dockerB1.row_count   39.0
2138  1614858000  2021-03-04 19:40:00 CST    MG01  trace.from_dockerB1.row_count   33.0
2141  1614858000  2021-03-04 19:40:00 CST    MG01  trace.from_dockerB2.row_count   27.0

[50 rows x 5 columns],    spike_timestamp cmdb_id  log_count  error_log_count
0       1614857820    MG01          0                0
1       1614857820    MG02          0                0
2       1614858000    MG01          0                0
3       1614858000    MG02          0                0, Empty DataFrame
Columns: [spike_timestamp, timestamp, datetime_utc8, cmdb_id, log_name, value]
Index: [], Empty DataFrame
Columns: [spike_timestamp, timestamp, datetime_utc8, cmdb_id, message]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```